# play

One game, one reply at a time. **Load** uses the same three choices as testing s2, plus an S1 file.

- **Full save** is an assembled directory under `weights/full/` (`s1.pt`, the readout, and `gemma/`), loaded with `S1S2.load`. The S1 dropdown is unused.
- **Gemma from here, localizer from here** is `GemmaS2.load_parts` plus the chosen S1 file, then `S1S2(s1, s2)`. When the Gemma directory has `coord_embed.pt`, that file is the coordinate code.
- **Coord checkpoint** is the LoRA plus `coord_embed.pt`, and the chosen S1 file. The readout is a fresh Localizer, so the points are not a trained readout. The status line says so.

A 3-point checkpoint fails to load and the log names `python -m neural_net.convert_to_sv`. **Refresh** rescans. Every reply is appended to `weights/s1s2_runs/<stamp>/turns.jsonl`, including the S1 action list. **S1 actions** only changes the conversation: off, a move is one summary line; on, that line is followed by the action list.

**New game** deals `new_multi_gold_game`. Gold defaults to random (0–3) and the opening to any, so a sealed empty room, a room with no gold, and a room with several golds all show up. The question box is filled with the default line: the move beginning on the first turn, then "Move to your target." after that. The user text includes the board report when the previous reply ended in `[MOVE]`. S1 stays dormant unless this reply ends in `[MOVE]`. Then S1 runs for 10 seconds, or 20 no-ops in a row, or until the agent exits. Walking out ends the game. A reply that ends in `[END_GAME]` ends the game without waking S1; that is the win on a sealed empty board and a loss otherwise. The board shows the end state.

**Agent turn** sends the question box once. **Auto-play** sends the default line, up to 12 turns, and stops when the game is over. The presets are the beginning lines, the two "your target" lines, and the direction and hour questions. Choosing one fills the box. It does not run the model.

Each turn loads this game's conversation from NAMS: the last 7 messages, plus up to 2 older ones by semantic search when the session has messages outside those 7. Every turn is stored, look or move. **New game** starts a new session. Neo4j has to be up (`bash scripts/vast_neo4j_launch.sh`).

Tokens of each turn are clickable. A click draws that token's **s** in blue and **v** in red on the frame S2 saw, not on the board after the move. Click the same token again to return to the end state. The token that completed `[MOVE]` is outlined in red. The token that completed `[END_GAME]` is outlined in green.

**Replay** rebuilds the latest move from its recorded actions and plays those frames, then returns to the end state. **End game** drops the session. The notepad is the in-memory `[REMEMBER]` notes. Control comes back after every turn. The trace file is written either way.

Under the log, **Flagged memories** lists every reply this game stored with `[FLAG failure: ...]`. Each line is the word FLAG. Click one to highlight it and open that memory below the list: the failed message labeled before, the flagging message labeled after, the picture each one saw, and the two Settings dicts as text. A flag on the first message shows an error placeholder instead of a prior message or picture. The box stays empty until a reply contains the flag line. The weights were not trained to write it.

**Tips** lists the saved tips from earlier games. A reply that contains `[TIP: one line]` saves that line at once and it shows up here. Edit, delete, or add a tip from the panel; the system prompt reloads from memory after each change, so the next turn sees it.

Run this on the GPU box. The cell needs the repo environment (`pygame`, `torch`, `ipywidgets`).


In [ ]:
from __future__ import annotations

import html
import io
import json
import os
import sys
import time
import traceback
from pathlib import Path

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets
import numpy as np
from IPython.display import HTML, display
from PIL import Image, ImageDraw

from agent.image_store import load_image
from agent.memory import (
    add_s2_tip,
    delete_core_tip,
    list_saved_s2_tips,
    load_flagged_memories,
    replace_core_tip,
)
from agent.modes import (
    S2_BEGINNING_MOVE_USER,
    S2_BEGINNING_USER,
    S2_LOOK_TARGET_USER,
    S2_MOVE_TARGET_USER,
)
from neural_net.loop import S1S2
from neural_net.paths import weights_root
from neural_net.render import canonical_frame
from neural_net.s1 import S1
from neural_net.s2 import GemmaS2
from neural_net.s2_coord_train import QUESTION_DIRECTION, QUESTION_HOUR
from neural_net.s2_session import S2GameSession
from neural_net.weight_catalog import (
    label as catalog_label,
    list_assembled,
    list_coord,
    list_gemma,
    list_localizer,
    list_s1,
)

BOARD_PX = 560
_COLOR = {"s": (30, 90, 255), "v": (255, 0, 0)}

state = {
    "net": None,
    "session": None,
    "turns": [],
    "end_frame": None,
    "selected": None,
    "busy": False,
    "ignore_through": 0,
    "gen": 0,
    "flags": [],
    "flag_id": None,
    "flag_gen": 0,
    "flag_ignore": 0,
    "tip_category": None,
}

FLAG_PX = 280


def _options(paths, empty):
    if not paths:
        return [(empty, "")]
    return [(catalog_label(Path(path)), str(path)) for path in paths]


def _keep(dropdown, options):
    selected = dropdown.value
    dropdown.options = options
    values = [value for _, value in options]
    dropdown.value = selected if selected in values else values[0]


def _scan():
    return (
        list_assembled(),
        list_gemma(),
        list_localizer(),
        list_coord(),
        list_s1(),
    )


def _say(text):
    log.append_stdout(text if text.endswith("\n") else text + "\n")


def _set_busy(flag):
    state["busy"] = flag
    for widget in controls:
        widget.disabled = flag


def _png(rgb):
    buf = io.BytesIO()
    Image.fromarray(np.ascontiguousarray(rgb), mode="RGB").save(buf, format="PNG")
    return buf.getvalue()


def _dot(draw, size, x, y, color):
    width, height = size
    px = float(x) * (width - 1)
    py = (1.0 - float(y)) * (height - 1)
    r = 8
    draw.ellipse((px - r, py - r, px + r, py + r), fill=color, outline=(0, 0, 0))


def _overlay(frame, token):
    image = Image.fromarray(np.ascontiguousarray(frame), mode="RGB")
    draw = ImageDraw.Draw(image)
    _piece, s_xy, v_xy, _eos, _move, _end = token
    if s_xy is not None:
        _dot(draw, image.size, s_xy[0], s_xy[1], _COLOR["s"])
    if v_xy is not None:
        _dot(draw, image.size, v_xy[0], v_xy[1], _COLOR["v"])
    return np.asarray(image)


def _show_board():
    selected = state["selected"]
    if selected is not None:
        turn_i, tok_i = selected
        turn = state["turns"][turn_i]
        board.value = _png(_overlay(turn.frame, turn.tokens[tok_i]))
        return
    frame = state["end_frame"]
    if frame is None:
        board.value = b""
        return
    board.value = _png(frame)


def _visible(piece):
    if piece == "":
        return "∅"
    return piece.replace(" ", "\u00a0")


def _token_markup():
    if not state["turns"]:
        return "<div class='fs-tokens'>Agent turn produces clickable tokens.</div>"
    gen = state["gen"]
    selected = state["selected"]
    blocks = []
    for turn_i, turn in enumerate(state["turns"]):
        parts = [f"<div class='fs-turn'>turn {turn_i + 1}</div>"]
        for tok_i, token in enumerate(turn.tokens):
            piece, _s, _v, eos, move_close, end_close = token
            cls = "fs-tok"
            if eos:
                cls += " fs-eos"
            if move_close:
                cls += " fs-move"
            if end_close:
                cls += " fs-end"
            if selected == (turn_i, tok_i):
                cls += " fs-on"
            parts.append(
                f"<span class='{cls}' data-gen='{gen}' "
                f"data-turn='{turn_i}' data-i='{tok_i}'>"
                f"{html.escape(_visible(piece))}</span>"
            )
        blocks.append("".join(parts))
    return "<div class='fs-tokens'>" + "<br>".join(blocks) + "</div>"


def _show_tokens():
    token_html.value = _token_markup()


def _move_note(turn):
    move = turn.move
    if move is None:
        return ""
    exited = "yes" if move.exited else "no"
    lines = [
        f"S1: stopped by {move.stopped_by}, {move.n_steps} steps, "
        f"collected {move.collected}, exited {exited}"
    ]
    if show_actions.value:
        lines.append(", ".join(move.actions) if move.actions else "(no actions)")
    return "<pre>" + html.escape("\n".join(lines)) + "</pre>"


def _show_log():
    blocks = []
    for turn_i, turn in enumerate(state["turns"]):
        blocks.append(
            f"<div class='fs-msg'><b>You, turn {turn_i + 1}</b>"
            f"<pre>{html.escape(turn.user_text)}</pre>"
            f"<b>Agent</b><pre>{html.escape(turn.text)}</pre>"
            f"{_move_note(turn)}</div>"
        )
    conv.value = "".join(blocks) if blocks else "<div class='fs-msg'>No turns yet.</div>"


def _show_notepad():
    session = state["session"]
    if session is None:
        notepad.value = "<pre>No game.</pre>"
        return
    notepad.value = f"<pre>{html.escape(session.notepad())}</pre>"


def _settings_text(raw):
    try:
        parsed = json.loads(raw)
    except ValueError:
        return raw
    return json.dumps(parsed, indent=2)


def _png_file(path):
    file = Path(load_image(path))
    if not file.is_file():
        raise FileNotFoundError(f"flagged board image is missing: {file}")
    return file.read_bytes()


def _render_flag_list():
    flags = state["flags"]
    if not flags:
        flag_list.value = "<div class='fs-flags'>No flagged memories.</div>"
        return
    gen = state["flag_gen"]
    selected = state["flag_id"]
    lines = []
    for index, item in enumerate(flags):
        cls = "fs-flag fs-on" if item["id"] == selected else "fs-flag"
        lines.append(
            f"<div class='{cls}' data-gen='{gen}' data-i='{index}'>FLAG</div>"
        )
    flag_list.value = "<div class='fs-flags'>" + "".join(lines) + "</div>"


def _render_flag_open():
    selected = state["flag_id"]
    item = next((row for row in state["flags"] if row["id"] == selected), None)
    if item is None:
        flag_open.layout.display = "none"
        flag_before.value = b""
        flag_after.value = b""
        return
    flag_message.value = (
        "<div class='fs-msg'><b>before</b> (the failed message)<pre>"
        + html.escape(item.get("before_message") or "")
        + "</pre><b>after</b> (the message that flagged it)<pre>"
        + html.escape(item.get("after_message") or item["content"])
        + "</pre></div>"
    )
    if item["before"].get("error"):
        flag_before.value = b""
        flag_before_cap.value = (
            "<b>before</b><pre>" + html.escape(item["before"]["error"]) + "</pre>"
        )
        flag_before_text.value = (
            "<div class='fs-msg'><b>before</b><pre>"
            + html.escape(item["before"].get("settings") or "[ERROR, no prior settings]")
            + "</pre></div>"
        )
    else:
        flag_before.value = _png_file(item["before"]["path"])
        flag_before_cap.value = "<b>before</b>"
        flag_before_text.value = (
            "<div class='fs-msg'><b>before</b><pre>"
            + html.escape(_settings_text(item["before"]["settings"]))
            + "</pre></div>"
        )
    flag_after.value = _png_file(item["after"]["path"])
    flag_after_cap.value = "<b>after</b>"
    flag_after_text.value = (
        "<div class='fs-msg'><b>after</b><pre>"
        + html.escape(_settings_text(item["after"]["settings"]))
        + "</pre></div>"
    )
    flag_open.layout.display = "flex"


def _show_flags():
    session = state["session"]
    if session is None or session.client is None:
        state["flags"] = []
    else:
        state["flags"] = session._run(load_flagged_memories(
            session.client, session.session_id,
        ))
    ids = [item["id"] for item in state["flags"]]
    if state["flag_id"] not in ids:
        state["flag_id"] = None
    state["flag_gen"] += 1
    _render_flag_list()
    _render_flag_open()


def _show_tips():
    session = state["session"]
    if session is None or session.client is None:
        tips_rows.children = (widgets.HTML("No game."),)
        return
    rows = session._run(list_saved_s2_tips(session.client))
    if not rows:
        tips_rows.children = (widgets.HTML("No saved tips."),)
        return
    children = []
    for row in rows:
        category = row["category"]
        text = row["text"]
        edit = widgets.Button(description="Edit", layout=widgets.Layout(width="70px"))
        delete = widgets.Button(description="Delete", layout=widgets.Layout(width="80px"))

        def _edit(_btn, category=category, text=text):
            if state["busy"]:
                return
            state["tip_category"] = category
            tip_editor.value = text
            tip_editing.value = f"<b>Editing</b> {html.escape(category)}"

        def _delete(_btn, category=category):
            if state["busy"]:
                return
            current = state["session"]
            if current is None or current.client is None:
                return
            current._run(delete_core_tip(current.client, category))
            if state["tip_category"] == category:
                state["tip_category"] = None
                tip_editor.value = ""
                tip_editing.value = "Not editing."
            current.reload_prompt()
            _show_tips()
            _say(f"deleted {category}")

        edit.on_click(_edit)
        delete.on_click(_delete)
        children.append(widgets.HBox([
            widgets.HTML(
                f"<b>{html.escape(category)}</b>"
                f"<pre>{html.escape(text)}</pre>"
            ),
            edit,
            delete,
        ]))
    tips_rows.children = tuple(children)


def _latest_move():
    for turn in reversed(state["turns"]):
        if turn.move is not None:
            return turn.move
    return None


def on_refresh(_):
    if state["busy"]:
        return
    assembled, gemma, localizer, coord, s1_paths = _scan()
    _keep(full_dd, _options(assembled, "(no full save)"))
    _keep(gemma_dd, _options(gemma, "(no Gemma)"))
    _keep(loc_dd, _options(localizer, "(no localizer)"))
    _keep(coord_dd, _options(coord, "(no coord checkpoint)"))
    _keep(s1_dd, _options(s1_paths, "(no S1 checkpoint)"))
    _say(
        f"{len(assembled)} full save(s), {len(gemma)} gemma, "
        f"{len(localizer)} localizer, {len(coord)} coord checkpoint(s), "
        f"{len(s1_paths)} S1 file(s) under {weights_root()}"
    )


def _load_s1(path):
    if not path:
        raise RuntimeError("pick an S1 checkpoint")
    actor = S1()
    actor.load_weights(path)
    return actor


def on_load(_):
    if state["busy"]:
        return
    mode = load_mode.value
    if mode == "full":
        full_path = full_dd.value
        if not full_path:
            status.value = "No full save selected."
            return
    elif mode == "parts":
        gemma_path = gemma_dd.value
        loc_path = loc_dd.value
        s1_path = s1_dd.value
        if not gemma_path or not loc_path:
            status.value = "Pick a Gemma directory and a localizer directory."
            return
        if not s1_path:
            status.value = "Pick an S1 checkpoint."
            return
    elif mode == "coord":
        coord_path = coord_dd.value
        s1_path = s1_dd.value
        if not coord_path:
            status.value = "No coord checkpoint selected."
            return
        if not s1_path:
            status.value = "Pick an S1 checkpoint."
            return
    else:
        raise RuntimeError(f"unknown load mode {mode!r}")
    _set_busy(True)
    status.value = "Loading S1 and S2."
    try:
        if mode == "full":
            state["net"] = S1S2.load(full_path)
            _say(f"loaded {full_path}")
            status.value = "New game, then Agent turn."
        elif mode == "parts":
            s2 = GemmaS2()
            s2.load_parts(gemma_path, loc_path)
            state["net"] = S1S2(s1=_load_s1(s1_path), s2=s2)
            _say(f"loaded gemma {gemma_path}  localizer {loc_path}  s1 {s1_path}")
            status.value = "New game, then Agent turn."
        else:
            s2 = GemmaS2()
            s2.load_coord_checkpoint(coord_path)
            state["net"] = S1S2(s1=_load_s1(s1_path), s2=s2)
            _say(f"loaded coord {coord_path}  s1 {s1_path}")
            status.value = (
                "Readout is a fresh Localizer. "
                "New game, then Agent turn."
            )
    except Exception:
        status.value = "Load failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_new(_):
    if state["busy"]:
        return
    if state["net"] is None:
        status.value = "Load a model first."
        return
    _set_busy(True)
    status.value = "Dealing a board."
    try:
        previous = state.get("session")
        if previous is not None:
            previous.close()
            state["session"] = None
        session = S2GameSession(
            state["net"],
            n_gold=gold.value,
            opening=str(opening.value),
        )
        state["session"] = session
        state["turns"] = []
        state["selected"] = None
        state["gen"] += 1
        state["end_frame"] = canonical_frame(session.game)
        question.value = session.next_question()
        _show_board()
        _show_tokens()
        _show_log()
        _show_notepad()
        _show_flags()
        _show_tips()
        _say(
            f"new game  gold={len(session.game.settings.gold)}  "
            f"opening={opening.value}"
        )
        _say(f"trace {session.turns_path}")
        status.value = "Agent turn sends the default question."
    except Exception:
        status.value = "New game failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_preset(change):
    if change["name"] != "value":
        return
    question.value = _PRESETS[change["new"]]


def _over_line(outcome):
    verdict = "won" if outcome["won"] else "lost"
    return f"Game over: {outcome['reason']}, {verdict}."


def _after_turn(result):
    session = state["session"]
    state["turns"].append(result)
    state["gen"] += 1
    state["end_frame"] = canonical_frame(session.game)
    _show_log()
    _show_tokens()
    _show_notepad()
    _show_board()
    _show_flags()
    _show_tips()
    if result.outcome is not None:
        status.value = _over_line(result.outcome)
    else:
        move = result.move
        if move is None:
            status.value = (
                f"Reply finished. No [MOVE]. "
                f"{result.gold_remaining} gold(s) remain."
            )
        else:
            status.value = (
                f"{move.n_steps} steps, {move.elapsed:.2f}s, "
                f"stopped by {move.stopped_by}, "
                f"gold eaten {move.collected}, "
                f"{result.gold_remaining} remaining"
                + (", agent exited." if result.exited else ".")
            )
    if not session.over:
        question.value = session.next_question()
    _say(status.value)


def on_turn(_):
    if state["busy"]:
        return
    session = state["session"]
    if session is None:
        status.value = "Start a game first."
        return
    if session.over:
        status.value = _over_line(session.outcome)
        return
    text = question.value.strip()
    if not text:
        status.value = "The question is empty."
        return
    _set_busy(True)
    status.value = "S2 is writing."
    state["selected"] = None
    try:
        _after_turn(session.turn(text))
    except Exception:
        status.value = "Agent turn failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_auto(_):
    if state["busy"]:
        return
    session = state["session"]
    if session is None:
        status.value = "Start a game first."
        return
    if session.over:
        status.value = _over_line(session.outcome)
        return
    _set_busy(True)
    status.value = "Auto-play."
    state["selected"] = None
    try:
        for result in session.play(max_turns=12):
            _after_turn(result)
        if not session.over:
            status.value = "Auto-play stopped at 12 turns. The game is still open."
            _say(status.value)
    except Exception:
        status.value = "Auto-play failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_replay(_):
    if state["busy"]:
        return
    session = state["session"]
    move = _latest_move()
    if session is None or move is None:
        status.value = "No move to replay."
        return
    _set_busy(True)
    state["selected"] = None
    status.value = "Replaying the latest move."
    try:
        n = move.n_steps + 1
        stride = max(1, n // 80)
        last = None
        for index, frame in enumerate(session.replay(move)):
            last = frame
            if index % stride == 0 or index + 1 == n:
                board.value = _png(frame)
                time.sleep(0.03)
        if last is not None:
            state["end_frame"] = last
        _show_board()
        status.value = (
            f"Replay finished. {move.n_steps} steps, "
            f"stopped by {move.stopped_by}."
        )
    except Exception:
        status.value = "Replay failed."
        _say(traceback.format_exc())
        raise
    finally:
        _set_busy(False)


def on_end(_):
    if state["busy"]:
        return
    previous = state.get("session")
    if previous is not None:
        previous.close()
    state["session"] = None
    state["turns"] = []
    state["end_frame"] = None
    state["selected"] = None
    state["gen"] += 1
    _show_board()
    _show_tokens()
    _show_log()
    _show_notepad()
    _show_flags()
    _show_tips()
    status.value = "Game ended."
    _say("game ended")


def on_pick(change):
    raw = change["new"]
    if not raw or raw.count(",") != 3:
        return
    seq_s, gen_s, turn_s, idx_s = raw.split(",")
    try:
        seq = int(seq_s)
        gen = int(gen_s)
        turn_i = int(turn_s)
        tok_i = int(idx_s)
    except ValueError:
        return
    if seq <= state["ignore_through"] or state["busy"]:
        return
    if gen != state["gen"]:
        return
    state["ignore_through"] = seq
    if turn_i < 0 or turn_i >= len(state["turns"]):
        return
    turn = state["turns"][turn_i]
    if tok_i < 0 or tok_i >= len(turn.tokens):
        return
    if state["selected"] == (turn_i, tok_i):
        state["selected"] = None
        readout.value = "Showing the board after the latest turn."
    else:
        state["selected"] = (turn_i, tok_i)
        piece, s_xy, v_xy, eos, move_close, end_close = turn.tokens[tok_i]
        readout.value = (
            f"turn {turn_i + 1} token {tok_i} &nbsp; {html.escape(piece)}<br>"
            f"s={s_xy} &nbsp; v={v_xy}"
            + ("<br>This token closed [MOVE]." if move_close else "")
            + ("<br>This token closed [END_GAME]." if end_close else "")
            + "<br>Drawn on the frame S2 saw."
        )
    _show_tokens()
    _show_board()


def on_flag(change):
    raw = change["new"]
    if not raw or raw.count(",") != 2:
        return
    seq_s, gen_s, idx_s = raw.split(",")
    try:
        seq = int(seq_s)
        gen = int(gen_s)
        index = int(idx_s)
    except ValueError:
        return
    if seq <= state["flag_ignore"] or state["busy"]:
        return
    if gen != state["flag_gen"]:
        return
    state["flag_ignore"] = seq
    if index < 0 or index >= len(state["flags"]):
        return
    chosen = state["flags"][index]["id"]
    if state["flag_id"] == chosen:
        state["flag_id"] = None
    else:
        state["flag_id"] = chosen
    _render_flag_list()
    _render_flag_open()


_PRESETS = {
    "begin_look": S2_BEGINNING_USER,
    "begin_move": S2_BEGINNING_MOVE_USER,
    "look_target": S2_LOOK_TARGET_USER,
    "move_target": S2_MOVE_TARGET_USER,
    "direction": QUESTION_DIRECTION,
    "hour": QUESTION_HOUR,
}

_assembled, _gemma, _localizer, _coord, _s1_paths = _scan()
_dd_style = {"description_width": "80px"}
_dd_layout = widgets.Layout(width="520px")
load_mode = widgets.RadioButtons(
    options=[
        ("Full save", "full"),
        ("Gemma from here, localizer from here", "parts"),
        ("Coord checkpoint", "coord"),
    ],
    value="full",
    description="Load:",
    style={"description_width": "50px"},
)
_full_opts = _options(_assembled, "(no full save)")
_gemma_opts = _options(_gemma, "(no Gemma)")
_loc_opts = _options(_localizer, "(no localizer)")
_coord_opts = _options(_coord, "(no coord checkpoint)")
_s1_opts = _options(_s1_paths, "(no S1 checkpoint)")
full_dd = widgets.Dropdown(
    options=_full_opts,
    value=_full_opts[0][1],
    description="Full:",
    style=_dd_style,
    layout=_dd_layout,
)
gemma_dd = widgets.Dropdown(
    options=_gemma_opts,
    value=_gemma_opts[0][1],
    description="Gemma:",
    style=_dd_style,
    layout=_dd_layout,
)
loc_dd = widgets.Dropdown(
    options=_loc_opts,
    value=_loc_opts[0][1],
    description="Localizer:",
    style=_dd_style,
    layout=_dd_layout,
)
coord_dd = widgets.Dropdown(
    options=_coord_opts,
    value=_coord_opts[0][1],
    description="Coord:",
    style=_dd_style,
    layout=_dd_layout,
)
s1_dd = widgets.Dropdown(
    options=_s1_opts,
    value=_s1_opts[0][1],
    description="S1:",
    style=_dd_style,
    layout=_dd_layout,
)
refresh_btn = widgets.Button(description="Refresh", icon="refresh")
load_btn = widgets.Button(description="Load", button_style="primary", icon="download")
gold = widgets.Dropdown(
    options=[("random", None)] + [(str(n), n) for n in (0, 1, 2, 3)],
    value=None,
    description="Gold:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="160px"),
)
opening = widgets.Dropdown(
    options=[("require exit", "require"), ("any", "any"), ("sealed", "forbid")],
    value="any",
    description="Opening:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="220px"),
)
new_btn = widgets.Button(description="New game", button_style="info")
end_btn = widgets.Button(description="End game")
preset = widgets.Dropdown(
    options=[
        ("Beginning, move", "begin_move"),
        ("Beginning, look", "begin_look"),
        ("Look at your target", "look_target"),
        ("Move to your target", "move_target"),
        ("Direction", "direction"),
        ("Hour", "hour"),
    ],
    value="begin_move",
    description="Preset:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="280px"),
)
question = widgets.Textarea(
    value=S2_BEGINNING_MOVE_USER,
    layout=widgets.Layout(width="100%", height="120px"),
)
turn_btn = widgets.Button(description="Agent turn", button_style="success", icon="play")
auto_btn = widgets.Button(description="Auto-play", button_style="warning")
replay_btn = widgets.Button(description="Replay", icon="refresh")
show_actions = widgets.Checkbox(
    value=False, description="S1 actions", indent=False,
)
status = widgets.HTML("<span style='font-family:sans-serif'>Load a checkpoint.</span>")
readout = widgets.HTML("Click a token to draw s and v on the frame S2 saw.")
legend = widgets.HTML(
    "<span style='color:#1e5aff'>\u25cf s</span> &nbsp; "
    "<span style='color:#ff0000'>\u25cf v</span> &nbsp; "
    "<span style='color:#c0392b'>[MOVE] closes the reply</span> &nbsp; "
    "<span style='color:#1a7f37'>[END_GAME] ends the game</span>"
)
board = widgets.Image(format="png", width=BOARD_PX, height=BOARD_PX)
token_html = widgets.HTML()
conv = widgets.HTML("<div class='fs-msg'>No turns yet.</div>")
notepad = widgets.HTML("<pre>No game.</pre>")
log = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc", height="180px", overflow="auto",
))
flag_list = widgets.HTML("<div class='fs-flags'>No flagged memories.</div>")
flag_message = widgets.HTML()
flag_before = widgets.Image(format="png", width=FLAG_PX, height=FLAG_PX)
flag_after = widgets.Image(format="png", width=FLAG_PX, height=FLAG_PX)
flag_before_cap = widgets.HTML("<b>before</b>")
flag_after_cap = widgets.HTML("<b>after</b>")
flag_before_text = widgets.HTML()
flag_after_text = widgets.HTML()
flag_open = widgets.VBox(
    [
        widgets.HTML("<b>Flagged memory</b>"),
        flag_message,
        widgets.HBox([
            widgets.VBox([flag_before_cap, flag_before]),
            widgets.VBox([flag_after_cap, flag_after]),
        ]),
        flag_before_text,
        flag_after_text,
    ],
    layout=widgets.Layout(
        border="1px solid #ccc",
        padding="8px",
        display="none",
        margin="12px 0 0 0",
    ),
)
flag_pick = widgets.Text(value="")
flag_pick.add_class("fs-flag-pick")
flag_pick.layout.height = "0px"
flag_pick.layout.visibility = "hidden"
flag_pick.layout.overflow = "hidden"
flag_pick.observe(on_flag, names="value")
tips_rows = widgets.VBox([widgets.HTML("No game.")])
tip_editing = widgets.HTML("Not editing.")
tip_editor = widgets.Textarea(
    placeholder="Edit the selected tip. One line.",
    layout=widgets.Layout(width="100%", height="60px"),
)
tip_save = widgets.Button(description="Save edit")
tip_new = widgets.Textarea(
    placeholder="Add a tip. One line.",
    layout=widgets.Layout(width="100%", height="60px"),
)
tip_add = widgets.Button(description="Add tip")


def on_save_edit(_):
    if state["busy"]:
        return
    session = state["session"]
    category = state["tip_category"]
    if session is None or session.client is None or not category:
        status.value = "Pick a tip to edit."
        return
    text = tip_editor.value.strip()
    if not text or "\n" in text:
        status.value = "A tip is one non-empty line."
        return
    session._run(replace_core_tip(session.client, category, text))
    session.reload_prompt()
    _show_tips()
    _say(f"updated {category}")


def on_add_tip(_):
    if state["busy"]:
        return
    session = state["session"]
    if session is None or session.client is None:
        status.value = "Start a game first."
        return
    text = tip_new.value.strip()
    if not text or "\n" in text:
        status.value = "A tip is one non-empty line."
        return
    category = session._run(add_s2_tip(session.client, text))
    session.reload_prompt()
    tip_new.value = ""
    _show_tips()
    _say(f"tip saved as {category}")


tip_save.on_click(on_save_edit)
tip_add.on_click(on_add_tip)
pick_box = widgets.Text(value="")
pick_box.add_class("fs-pick")
pick_box.layout.height = "0px"
pick_box.layout.visibility = "hidden"
pick_box.layout.overflow = "hidden"
pick_box.observe(on_pick, names="value")
preset.observe(on_preset, names="value")


def on_actions(change):
    if change["name"] != "value" or state["busy"]:
        return
    _show_log()


show_actions.observe(on_actions, names="value")


controls = [
    load_mode, full_dd, gemma_dd, loc_dd, coord_dd, s1_dd,
    refresh_btn, load_btn, gold, opening, new_btn, end_btn,
    preset, question, turn_btn, auto_btn, replay_btn, show_actions,
]
refresh_btn.on_click(on_refresh)
load_btn.on_click(on_load)
new_btn.on_click(on_new)
end_btn.on_click(on_end)
turn_btn.on_click(on_turn)
auto_btn.on_click(on_auto)
replay_btn.on_click(on_replay)

_say(
    f"{len(_assembled)} full save(s), {len(_gemma)} gemma, "
    f"{len(_localizer)} localizer, {len(_coord)} coord checkpoint(s), "
    f"{len(_s1_paths)} S1 file(s) under {weights_root()}"
)
_show_tokens()
_show_flags()
_show_tips()

display(HTML(
    """
<style>
.fs-tokens {
  white-space: pre-wrap;
  line-height: 1.8;
  border: 1px solid #ccc;
  padding: 8px;
  max-height: 220px;
  overflow-y: auto;
  font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
}
.fs-tok { cursor: pointer; border-radius: 3px; }
.fs-tok:hover { background: #e8eefc; }
.fs-tok.fs-on { background: #ffe08a; }
.fs-eos { opacity: 0.55; }
.fs-move { box-shadow: inset 0 0 0 2px #c0392b; }
.fs-end { box-shadow: inset 0 0 0 2px #1a7f37; }
.fs-turn { font-family: sans-serif; font-size: 12px; color: #666; }
.fs-msg pre {
  white-space: pre-wrap;
  font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
  font-size: 13px;
}
.fs-log {
  max-height: 280px;
  overflow-y: auto;
  border: 1px solid #ccc;
  padding: 8px;
}
.fs-flags {
  border: 1px solid #ccc;
  padding: 8px;
  max-height: 140px;
  overflow-y: auto;
  font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
  line-height: 1.6;
}
.fs-flag { cursor: pointer; display: block; }
.fs-flag:hover { background: #e8eefc; }
.fs-flag.fs-on { background: #ffe08a; }
</style>
<script>
(function () {
  if (window.__fsTokenClick) { return; }
  window.__fsTokenClick = true;
  window.addEventListener("click", function (ev) {
    var span = ev.target && ev.target.closest && ev.target.closest(".fs-tok");
    if (!span) { return; }
    var box = document.querySelector(".fs-pick input");
    if (!box) { return; }
    window.__fsPickN = (window.__fsPickN || 0) + 1;
    var gen = span.getAttribute("data-gen") || "0";
    var turn = span.getAttribute("data-turn") || "0";
    var i = span.getAttribute("data-i");
    box.value = window.__fsPickN + "," + gen + "," + turn + "," + i;
    box.dispatchEvent(new Event("input", { bubbles: true }));
  }, true);
})();
(function () {
  if (window.__fsFlagClick) { return; }
  window.__fsFlagClick = true;
  window.addEventListener("click", function (ev) {
    var row = ev.target && ev.target.closest && ev.target.closest(".fs-flag");
    if (!row) { return; }
    var box = document.querySelector(".fs-flag-pick input");
    if (!box) { return; }
    window.__fsFlagN = (window.__fsFlagN || 0) + 1;
    var gen = row.getAttribute("data-gen") || "0";
    var i = row.getAttribute("data-i");
    box.value = window.__fsFlagN + "," + gen + "," + i;
    box.dispatchEvent(new Event("input", { bubbles: true }));
  }, true);
})();
</script>
"""
))
display(widgets.VBox([
    load_mode,
    widgets.HBox([full_dd, refresh_btn, load_btn]),
    widgets.HBox([gemma_dd, loc_dd]),
    widgets.HBox([coord_dd, s1_dd]),
    widgets.HBox([gold, opening, new_btn, end_btn]),
    preset,
    question,
    widgets.HBox([turn_btn, auto_btn, replay_btn, show_actions, legend]),
    status,
    readout,
    widgets.HBox([
        board,
        widgets.VBox([
            widgets.HTML("<b>Notepad</b>"),
            notepad,
            widgets.HTML("<b>Conversation</b>"),
            widgets.VBox([conv], layout=widgets.Layout(
                height="280px", overflow="auto", border="1px solid #ccc",
            )),
        ], layout=widgets.Layout(flex="1 1 auto")),
    ], layout=widgets.Layout(align_items="flex-start")),
    token_html,
    log,
    widgets.HTML("<b>Flagged memories</b>"),
    flag_list,
    flag_open,
    widgets.HTML("<b>Tips</b>"),
    tips_rows,
    tip_editing,
    tip_editor,
    tip_save,
    tip_new,
    tip_add,
    pick_box,
    flag_pick,
]))
